In [4]:
"""
compute_deltas.py
Reads stage CSVs, item_coverage.xlsx, and model-level test CSVs.
Emits the results tables used in the paper plus the RQ4 loss-curve figure.
"""

import csv
import ast
from pathlib import Path
from google.colab import drive
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

drive.mount('/content/drive')

# ---------- Configuration ----------
PATHS = {
    "stage1": "/content/drive/MyDrive/results/stage1_results_full.csv",
    "stage2": "/content/drive/MyDrive/results/stage2_results_full.csv",
    "stage3": "/content/drive/MyDrive/results/stage3_results_full.csv",
    "stage4": "/content/drive/MyDrive/results/stage4_results_full.csv",
    "stage5": "/content/drive/MyDrive/results/stage5_results_full.csv",
    "stage6": "/content/drive/MyDrive/results/stage6_results_full.csv",
    "coverage": "/content/drive/MyDrive/results/item_coverage.xlsx",
}

MODEL_LEVEL_PATHS = {
    "avg":    "/content/drive/MyDrive/results/Model_Level_Results/test_results_amazon_videogames.csv",
    "toys":   "/content/drive/MyDrive/results/Model_Level_Results/test_results_amazon_toys_and_games.csv",
    "ml1m":   "/content/drive/MyDrive/results/Model_Level_Results/test_results_movielens_1m.csv",
    "lastfm": "/content/drive/MyDrive/results/Model_Level_Results/test_results_lastfm_1k.csv",
}

DATASETS = ["amazon_videogames", "amazon_toys_and_games", "movielens_1m", "lastfm_1k"]

SPARSITY_ORDER = [
    ("Toys",   "amazon_toys_and_games"),
    ("AVG",    "amazon_videogames"),
    ("LastFM", "lastfm_1k"),
    ("ML-1M",  "movielens_1m"),
]

RQ_ROWS_4 = [
    ("Mamba4Rec", "Mamba4Rec", "Mamba4Rec_CL", "CE"),
    ("Mamba4Rec", "Mamba4Rec", "Mamba4Rec_CL", "BPR"),
    ("SIGMA",     "SIGMA",     "SIGMA_CL",     "CE"),
    ("SIGMA",     "SIGMA",     "SIGMA_CL",     "BPR"),
]

SPARSITY_NOTES = {
    "Toys":   "99.99\\%",
    "AVG":    "99.95\\%",
    "LastFM": "97.30\\%",
    "ML-1M":  "95.53\\%",
}

STAGE_EXTRA_COLUMNS = ["train_losses", "epoch_times", "valid_metrics"]

# ---------- Load ----------

def load_rows(path):
    p = Path(path)
    if p.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, dtype=str, keep_default_na=False)
        rows_raw = df.to_dict(orient="records")
    else:
        with open(path, encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f)
            header = next(reader)
            data_rows = list(reader)

        if data_rows and len(data_rows[0]) > len(header):
            for i, name in enumerate(STAGE_EXTRA_COLUMNS):
                if len(header) + i < len(data_rows[0]):
                    header.append(name)
            print(f"[info] {p.name}: extended header from {len(header)-3} to {len(header)} columns")

        rows_raw = [dict(zip(header, r)) for r in data_rows]

    out = []
    for r in rows_raw:
        cleaned = {str(k).strip(): (v.strip() if isinstance(v, str) else v) for k, v in r.items()}
        if str(cleaned.get("status", "")).lower() == "failed":
            continue
        out.append(cleaned)
    return out

def load_all():
    data = {}
    for key, path in PATHS.items():
        if not Path(path).exists():
            print(f"[warn] {path} not found — skipping")
            continue
        data[key] = load_rows(path)
    return data

def load_model_level():
    rows = []
    for name, path in MODEL_LEVEL_PATHS.items():
        p = Path(path)
        if not p.exists():
            print(f"[warn] {path} not found — check the folder name")
            continue
        r = load_rows(path)
        print(f"[info] {name}: {len(r)} rows loaded from {p.name}")
        rows.extend(r)
    print(f"[info] model-level total: {len(rows)} rows\n")
    return rows

# ---------- Helpers ----------

def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def find_row(rows, model, loss, cl_loss, dataset):
    matches = [
        r for r in rows
        if r.get("model", "").strip() == model
        and r.get("loss_type", "").strip() == loss
        and r.get("cl_loss_type", "").strip() == cl_loss
        and r.get("dataset", "").strip() == dataset
    ]
    if not matches:
        return None
    if len(matches) > 1:
        matches.sort(key=lambda r: int(r.get("num_epochs") or 0), reverse=True)
        print(f"[warn] duplicate rows for {model}/{loss}/{cl_loss}/{dataset}, using longest run")
    return matches[0]

def delta_pct(baseline_val, cl_val):
    if baseline_val is None or cl_val is None or baseline_val == 0:
        return None
    return (cl_val - baseline_val) / baseline_val * 100

def mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else None

def fmt_pct(v, bold=False):
    if v is None:
        return "---"
    s = f"{v:+.1f}\\%"
    return f"$\\mathbf{{{s}}}$" if bold else f"${s}$"

def _fmt4(v):
    return f"{v:.4f}" if v is not None else "---"

def _fmt2(v):
    return f"{v:.2f}" if v is not None else "---"

def _fmt_int(v):
    return f"{v:.0f}" if v is not None else "---"

def parse_list(s):
    if not s or not isinstance(s, str) or not s.strip().startswith("["):
        return []
    s = s.strip().replace("-Infinity", "None").replace("Infinity", "None")
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

# ---------- Sparsity-ordered helpers ----------

def _ds_vals(rows, base_model, loss, metric):
    out = []
    for _, ds in SPARSITY_ORDER:
        r = find_row(rows, base_model, loss, "none", ds)
        out.append(to_float(r.get(metric)) if r else None)
    return out

def _ds_cl_vals(rows, cl_model, loss, metric):
    """Return [Toys, AVG, LastFM, ML-1M] absolute CL values for a config."""
    out = []
    for _, ds in SPARSITY_ORDER:
        vals = []
        for cl_type in ["info_nce", "dcl"]:
            r = find_row(rows, cl_model, loss, cl_type, ds)
            if r is not None:
                v = to_float(r.get(metric))
                if v is not None:
                    vals.append(v)
        out.append(mean(vals) if vals else None)
    return out

def _merged_cl_row(rows, base_model, cl_model, loss, metric):
    out = []
    for _, ds in SPARSITY_ORDER:
        base = find_row(rows, base_model, loss, "none", ds)
        if base is None:
            out.append(None); continue
        b = to_float(base.get(metric))
        if b is None or b == 0:
            out.append(None); continue
        vals = []
        for cl_type in ["info_nce", "dcl"]:
            cl = find_row(rows, cl_model, loss, cl_type, ds)
            if cl is not None:
                d = delta_pct(b, to_float(cl.get(metric)))
                if d is not None:
                    vals.append(d)
        out.append(mean(vals) if vals else None)
    return out

def _merged_cov_row(cov_rows, base_model, cl_model, loss, metric="coverage@10"):
    out = []
    for _, ds in SPARSITY_ORDER:
        base = find_row(cov_rows, base_model, loss, "none", ds)
        if base is None:
            out.append(None); continue
        b = to_float(base.get(metric))
        if b is None or b == 0:
            out.append(None); continue
        vals = []
        for cl_type in ["info_nce", "dcl"]:
            cl = find_row(cov_rows, cl_model, loss, cl_type, ds)
            if cl is not None:
                d = delta_pct(b, to_float(cl.get(metric)))
                if d is not None:
                    vals.append(d)
        out.append(mean(vals) if vals else None)
    return out

def _row_mean(vals):
    return mean([v for v in vals if v is not None])

def _mean_delta_across_datasets(rows, base_model, cl_model, loss, metric):
    row = _merged_cl_row(rows, base_model, cl_model, loss, metric)
    return mean([v for v in row if v is not None])

def _mean_cov_delta_across_datasets(cov_rows, base_model, cl_model, loss):
    row = _merged_cov_row(cov_rows, base_model, cl_model, loss)
    return mean([v for v in row if v is not None])

def _attention_vs_sasrec(rows, cl_model, metric, base_model="SASRec"):
    vals = []
    for ds in DATASETS:
        base = find_row(rows, base_model, "default", "none", ds)
        cl = find_row(rows, cl_model, "default", "none", ds)
        if base is None or cl is None:
            continue
        d = delta_pct(to_float(base.get(metric)), to_float(cl.get(metric)))
        if d is not None:
            vals.append(d)
    return mean(vals)

def _attention_cov_vs_sasrec(cov_rows, cl_model, base_model="SASRec"):
    vals = []
    for ds in DATASETS:
        base = find_row(cov_rows, base_model, "default", "none", ds)
        cl = find_row(cov_rows, cl_model, "default", "none", ds)
        if base is None or cl is None:
            continue
        d = delta_pct(to_float(base.get("coverage@10")), to_float(cl.get("coverage@10")))
        if d is not None:
            vals.append(d)
    return mean(vals)

# ---------- RQ1 — Accuracy ----------

def emit_rq1_compact(data, ml_rows, filename="table_rq1.tex"):
    """RQ1 — Accuracy: Baseline, CL per dataset, and Mean Delta%."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ1 — Recommendation accuracy with and without contrastive regularization.}",
        r"\label{tab:rq1_compact}",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{3pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll cccc cccc ccc}",
        r"\toprule",
        r"& & \multicolumn{4}{c}{Baseline NDCG@10} & \multicolumn{4}{c}{CL NDCG@10} & \multicolumn{3}{c}{Mean \(\Delta\%\)} \\",
        r"\cmidrule(lr){3-6} \cmidrule(lr){7-10} \cmidrule(lr){11-13}",
        r"Backbone & Loss & Toys & AVG & LastFM & ML-1M & Toys & AVG & LastFM & ML-1M & NDCG@10 & Hit@10 & MRR@10 \\",
        r"\midrule",
    ]

    for panel_label, rows in [
        (r"\textit{A. Data-level regularization}", all_rows),
        (r"\textit{B. Model-level regularization (preliminary)}", ml_rows)
    ]:
        L.append(rf"\multicolumn{{13}}{{l}}{{{panel_label}}} \\")
        for backbone, base_model, cl_model, loss in RQ_ROWS_4:
            base_vals = _ds_vals(rows, base_model, loss, "ndcg@10")
            cl_vals   = _ds_cl_vals(rows, cl_model, loss, "ndcg@10")

            ndcg = _merged_cl_row(rows, base_model, cl_model, loss, "ndcg@10")
            hit  = _merged_cl_row(rows, base_model, cl_model, loss, "hit@10")
            mrr  = _merged_cl_row(rows, base_model, cl_model, loss, "mrr@10")

            L.append(f"{backbone} & {loss} & "
                     + " & ".join(_fmt4(v) for v in base_vals)
                     + " & " + " & ".join(_fmt4(v) for v in cl_vals)
                     + f" & {fmt_pct(_row_mean(ndcg))}"
                     + f" & {fmt_pct(_row_mean(hit))}"
                     + f" & {fmt_pct(_row_mean(mrr))} \\\\")
        if panel_label.startswith(r"\textit{A"):
            L.append(r"\midrule")

    L += [
        r"\bottomrule",
        r"\end{tabular}}",
        r"\vspace{2pt}",
        r"\parbox{\textwidth}{\footnotesize \raggedright Note: The Mean \(\Delta\%\) is calculated by averaging the per-dataset percentage changes between the Baseline and CL values.}",
        r"\end{table}",
    ]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


# ---------- RQ2 — Geometry ----------

def emit_rq2_compact(data, ml_rows, filename="table_rq2.tex"):
    """RQ2 — Geometry: Baseline, CL per dataset, and Mean Delta%."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ2 — Representation geometry with and without contrastive regularization.}",
        r"\label{tab:rq2_compact}",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{3pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll cccc cccc cc}",
        r"\toprule",
        r"& & \multicolumn{4}{c}{Baseline Isotropy} & \multicolumn{4}{c}{CL Isotropy} & \multicolumn{2}{c}{Mean \(\Delta\%\)} \\",
        r"\cmidrule(lr){3-6} \cmidrule(lr){7-10} \cmidrule(lr){11-12}",
        r"Backbone & Loss & Toys & AVG & LastFM & ML-1M & Toys & AVG & LastFM & ML-1M & Isotropy & Eff. Rank \\",
        r"\midrule",
    ]

    for panel_label, rows in [
        (r"\textit{A. Data-level regularization}", all_rows),
        (r"\textit{B. Model-level regularization (preliminary)}", ml_rows)
    ]:
        L.append(rf"\multicolumn{{12}}{{l}}{{{panel_label}}} \\")
        for backbone, base_model, cl_model, loss in RQ_ROWS_4:
            base_vals = _ds_vals(rows, base_model, loss, "isotropy")
            cl_vals   = _ds_cl_vals(rows, cl_model, loss, "isotropy")

            iso = _merged_cl_row(rows, base_model, cl_model, loss, "isotropy")
            er  = _merged_cl_row(rows, base_model, cl_model, loss, "effective_rank")

            L.append(f"{backbone} & {loss} & "
                     + " & ".join(_fmt4(v) for v in base_vals)
                     + " & " + " & ".join(_fmt4(v) for v in cl_vals)
                     + f" & {fmt_pct(_row_mean(iso))}"
                     + f" & {fmt_pct(_row_mean(er))} \\\\")
        if panel_label.startswith(r"\textit{A"):
            L.append(r"\midrule")

    L += [
        r"\bottomrule",
        r"\end{tabular}}",
        r"\vspace{2pt}",
        r"\parbox{\textwidth}{\footnotesize \raggedright Note: The Mean \(\Delta\%\) is calculated by averaging the per-dataset percentage changes between the Baseline and CL values.}",
        r"\end{table}",
    ]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


# ---------- RQ3 — Architecture ----------

def emit_rq3_compact(data, ml_rows, filename="table_rq3.tex"):
    """RQ3 — Architecture comparison."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ3 — Contrastive learning effect direction: SSM vs Attention. Mean $\Delta\%$ across four datasets. Values > 0 indicate CL improved the metric relative to the appropriate baseline.}",
        r"\label{tab:rq3_compact}",
        r"\small",
        r"\setlength{\tabcolsep}{10pt}",
        r"\begin{tabular}{l c c}",
        r"\toprule",
        r"Config & Mean $\Delta\%$ NDCG@10 & Mean $\Delta\%$ Isotropy \\",
        r"\midrule",
        r"\multicolumn{3}{l}{\textit{A. SSM + CL vs own baseline (data-level; CE or BPR, matched)}} \\",
    ]

    for backbone, base_model, cl_model, loss in RQ_ROWS_4:
        ndcg = _mean_delta_across_datasets(all_rows, base_model, cl_model, loss, "ndcg@10")
        iso  = _mean_delta_across_datasets(all_rows, base_model, cl_model, loss, "isotropy")
        L.append(f"{backbone} {loss} + CL & {fmt_pct(ndcg)} & {fmt_pct(iso)} \\\\")

    L.append(r"\midrule")
    L.append(r"\multicolumn{3}{l}{\textit{B. SSM + CL vs own baseline (model-level; preliminary)}} \\")
    for backbone, base_model, cl_model, loss in RQ_ROWS_4:
        ndcg = _mean_delta_across_datasets(ml_rows, base_model, cl_model, loss, "ndcg@10")
        iso  = _mean_delta_across_datasets(ml_rows, base_model, cl_model, loss, "isotropy")
        L.append(f"{backbone} {loss} + CL & {fmt_pct(ndcg)} & {fmt_pct(iso)} \\\\")

    L.append(r"\midrule")
    L.append(r"\multicolumn{3}{l}{\textit{C. Attention CL vs SASRec}} \\")
    for cl_model in ["CL4SRec", "DuoRec"]:
        ndcg = _attention_vs_sasrec(all_rows, cl_model, "ndcg@10")
        iso  = _attention_vs_sasrec(all_rows, cl_model, "isotropy")
        L.append(f"{cl_model} vs SASRec & {fmt_pct(ndcg)} & {fmt_pct(iso)} \\\\")

    ssm_ndcg = mean([_mean_delta_across_datasets(all_rows, b, c, l, "ndcg@10")
                     for b, _, c, l in RQ_ROWS_4])
    ssm_iso  = mean([_mean_delta_across_datasets(all_rows, b, c, l, "isotropy")
                     for b, _, c, l in RQ_ROWS_4])

    att_ndcg = mean([_attention_vs_sasrec(all_rows, m, "ndcg@10") for m in ["CL4SRec", "DuoRec"]])
    att_iso  = mean([_attention_vs_sasrec(all_rows, m, "isotropy") for m in ["CL4SRec", "DuoRec"]])

    def verdict(a, s):
        if a is None or s is None:
            return "---"
        return "Yes" if (a * s) > 0 else "No"

    L.append(r"\midrule")
    L.append(r"\multicolumn{3}{l}{\textit{D. Effect direction consistent across architecture families?}} \\")
    for label, ssm, att in [("NDCG@10", ssm_ndcg, att_ndcg),
                            ("Isotropy", ssm_iso, att_iso)]:
        L.append(f"{label} & SSM: {fmt_pct(ssm)} / Attn: {fmt_pct(att)} & {verdict(ssm, att)} \\\\")

    L += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\vspace{2pt}",
        r"\footnotesize Model-level values are preliminary (10-epoch runs). Panel D compares data-level SSM results (Panel A) against attention results (Panel C).",
        r"\end{table}",
    ]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


# ---------- RQ4 — Stability ----------

def emit_rq4_compact(data, filename="table_rq4.tex"):
    """RQ4 — Stability by interaction density."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])

    def _mean_for(cl_model, loss, cl_types, ds_id):
        eps, times = [], []
        for ct in cl_types:
            r = find_row(all_rows, cl_model, loss, ct, ds_id)
            if r is None:
                continue
            ep = to_float(r.get("num_epochs"))
            if ep is not None: eps.append(ep)
            t = to_float(r.get("avg_epoch_time_sec"))
            if t is not None: times.append(t)
        return mean(eps), mean(times)

    def _config_vals(base_model, cl_model, ds_id):
        bl_ep, bl_t, cl_ep, cl_t = [], [], [], []
        for loss in ["CE", "BPR"]:
            e1, t1 = _mean_for(base_model, loss, ["none"], ds_id)
            if e1 is not None: bl_ep.append(e1)
            if t1 is not None: bl_t.append(t1)
            e2, t2 = _mean_for(cl_model, loss, ["info_nce", "dcl"], ds_id)
            if e2 is not None: cl_ep.append(e2)
            if t2 is not None: cl_t.append(t2)
        return (mean(bl_ep), mean(bl_t), mean(cl_ep), mean(cl_t))

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ4 — Training behaviour by interaction density. Epochs and mean epoch time (seconds) reported per configuration; overhead columns express +CL time as a multiple of the matched baseline. Values averaged across CE and BPR runs.}",
        r"\label{tab:rq4_compact}",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{4pt}",
        r"\begin{tabular}{l cc cc cc cc cc}",
        r"\toprule",
        r"& \multicolumn{2}{c}{Mamba BL} & \multicolumn{2}{c}{Mamba +CL} "
        r"& \multicolumn{2}{c}{SIGMA BL} & \multicolumn{2}{c}{SIGMA +CL} "
        r"& \multicolumn{2}{c}{Overhead} \\",
        r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11}",
        r"Dataset (sparsity) & Eps & T(s) & Eps & T(s) & Eps & T(s) & Eps & T(s) & Mamba & SIGMA \\",
        r"\midrule",
    ]

    for ds_label, ds_id in SPARSITY_ORDER:
        ds_display = f"{ds_label} ({SPARSITY_NOTES[ds_label]})"

        m_bl_ep, m_bl_t, m_cl_ep, m_cl_t = _config_vals("Mamba4Rec", "Mamba4Rec_CL", ds_id)
        s_bl_ep, s_bl_t, s_cl_ep, s_cl_t = _config_vals("SIGMA",     "SIGMA_CL",     ds_id)

        m_ov = (m_cl_t / m_bl_t) if (m_cl_t and m_bl_t and m_bl_t > 0) else None
        s_ov = (s_cl_t / s_bl_t) if (s_cl_t and s_bl_t and s_bl_t > 0) else None

        L.append(
            f"{ds_display} & "
            + f"{_fmt_int(m_bl_ep)} & {_fmt2(m_bl_t)} & "
            + f"{_fmt_int(m_cl_ep)} & {_fmt2(m_cl_t)} & "
            + f"{_fmt_int(s_bl_ep)} & {_fmt2(s_bl_t)} & "
            + f"{_fmt_int(s_cl_ep)} & {_fmt2(s_cl_t)} & "
            + (f"${m_ov:.2f}\\times$" if m_ov is not None else "---") + " & "
            + (f"${s_ov:.2f}\\times$" if s_ov is not None else "---") + r" \\"
        )

    L += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\vspace{2pt}",
        r"\footnotesize BL = baseline. Eps = mean epochs; T(s) = mean epoch time in seconds. Overhead is expressed relative to the matched baseline's epoch time.",
        r"\end{table}",
    ]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


def loss_increase_pct(train_losses_str):
    ls = parse_list(train_losses_str)
    if len(ls) < 2:
        return None
    inc = sum(1 for i in range(1, len(ls))
              if ls[i] is not None and ls[i-1] is not None and ls[i] > ls[i-1])
    return inc / (len(ls) - 1) * 100


def emit_rq4_stability_table(data, filename="table_rq4_stability.tex"):
    """RQ4 — Training-loss volatility."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5"] if k in data), [])

    def mean_inc(model, loss, cl_types):
        vals = []
        for _, ds in SPARSITY_ORDER:
            for ct in cl_types:
                r = find_row(all_rows, model, loss, ct, ds)
                if r is None:
                    continue
                v = loss_increase_pct(r.get("train_losses", ""))
                if v is not None:
                    vals.append(v)
        return mean(vals)

    CONFIGS = [
        ("Mamba4Rec baseline", "Mamba4Rec",    ["none"]),
        ("Mamba4Rec + CL",     "Mamba4Rec_CL", ["info_nce", "dcl"]),
        ("SIGMA baseline",     "SIGMA",        ["none"]),
        ("SIGMA + CL",         "SIGMA_CL",     ["info_nce", "dcl"]),
    ]

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ4 — Training-loss volatility by primary loss objective. Values report the mean percentage of epochs in which training loss increased relative to the previous epoch, averaged across four datasets and (for +CL rows) across InfoNCE and DCL.}",
        r"\label{tab:rq4_stability}",
        r"\small",
        r"\setlength{\tabcolsep}{12pt}",
        r"\begin{tabular}{l c c}",
        r"\toprule",
        r"Config & CE & BPR \\",
        r"\midrule",
    ]
    for label, model, cl_types in CONFIGS:
        ce  = mean_inc(model, "CE",  cl_types)
        bpr = mean_inc(model, "BPR", cl_types)
        ce_s  = f"{ce:.1f}\\%"  if ce  is not None else "---"
        bpr_s = f"{bpr:.1f}\\%" if bpr is not None else "---"
        L.append(f"{label} & {ce_s} & {bpr_s} \\\\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


def emit_rq4_loss_curves(data, filename="fig_rq4_loss_curves.pdf"):
    """RQ4 training-loss curves for Mamba4Rec on one dataset."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])

    DS_ID = "lastfm_1k"
    DS_LABEL = "LastFM-1K"

    SERIES = [
        ("Mamba4Rec CE baseline",  "Mamba4Rec",    "CE",  "none",     "tab:blue",   "-"),
        ("Mamba4Rec CE + InfoNCE", "Mamba4Rec_CL", "CE",  "info_nce", "tab:blue",   "--"),
        ("Mamba4Rec BPR baseline", "Mamba4Rec",    "BPR", "none",     "tab:orange", "-"),
        ("Mamba4Rec BPR + InfoNCE","Mamba4Rec_CL", "BPR", "info_nce", "tab:orange", "--"),
    ]

    fig, ax = plt.subplots(figsize=(7, 3.6))

    for label, model, loss, cl_type, color, ls in SERIES:
        row = find_row(all_rows, model, loss, cl_type, DS_ID)
        if row is None:
            print(f"[warn] missing row for {label} — skipping")
            continue
        losses = parse_list(row.get("train_losses", ""))
        if not losses:
            print(f"[warn] empty train_losses for {label}")
            continue
        xs = list(range(1, len(losses) + 1))
        ax.plot(xs, losses, label=label, color=color, linestyle=ls,
                linewidth=1.4, alpha=0.9)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss")
    ax.set_title(f"Mamba4Rec training-loss trajectories ({DS_LABEL})", fontsize=10)
    ax.grid(True, alpha=0.3, linestyle=":")
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, loc="upper right", frameon=True, framealpha=0.95)

    plt.tight_layout()
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Wrote {filename}\n")


# ---------- RQ5 — Coverage ----------

def emit_rq5_compact(data, filename="table_rq5.tex"):
    """RQ5 — Coverage."""
    cov_rows = data.get("coverage", [])

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{RQ5 — Item coverage at K = 10. Direct values (Panels A and C) and percentage change relative to the corresponding baseline (Panels B and D).}",
        r"\label{tab:rq5_compact}",
        r"\small",
        r"\setlength{\tabcolsep}{6pt}",
        r"\begin{tabular}{l l c c c c c}",
        r"\toprule",
        r"Config & Loss & Toys & AVG & LastFM & ML-1M & Mean \\",
        r"\midrule",
        r"\multicolumn{7}{l}{\textit{A. SSM baselines — Coverage@10 (absolute)}} \\",
    ]

    for model, loss in [("Mamba4Rec", "CE"), ("Mamba4Rec", "BPR"),
                        ("SIGMA",     "CE"), ("SIGMA",     "BPR")]:
        vals = _ds_vals(cov_rows, model, loss, "coverage@10")
        L.append(f"{model} & {loss} & " + " & ".join(_fmt4(v) for v in vals)
                 + f" & {_fmt4(_row_mean(vals))} \\\\")

    L.append(r"\midrule")
    L.append(r"\multicolumn{7}{l}{\textit{B. SSM + CL — $\Delta\%$ vs own baseline (merged InfoNCE + DCL)}} \\")
    for backbone, base_model, cl_model, loss in RQ_ROWS_4:
        row = _merged_cov_row(cov_rows, base_model, cl_model, loss)
        L.append(f"{backbone} & {loss} & " + " & ".join(fmt_pct(v) for v in row)
                 + f" & {fmt_pct(_row_mean(row))} \\\\")

    L.append(r"\midrule")
    L.append(r"\multicolumn{7}{l}{\textit{C. Attention baseline — Coverage@10 (absolute)}} \\")
    sasrec_vals = _ds_vals(cov_rows, "SASRec", "default", "coverage@10")
    L.append("SASRec & CE & " + " & ".join(_fmt4(v) for v in sasrec_vals)
             + f" & {_fmt4(_row_mean(sasrec_vals))} \\\\")

    L.append(r"\midrule")
    L.append(r"\multicolumn{7}{l}{\textit{D. Attention CL vs SASRec — $\Delta\%$}} \\")
    for cl_model in ["CL4SRec", "DuoRec"]:
        row = []
        for _, ds in SPARSITY_ORDER:
            base = find_row(cov_rows, "SASRec", "default", "none", ds)
            cl = find_row(cov_rows, cl_model, "default", "none", ds)
            if base is None or cl is None:
                row.append(None); continue
            row.append(delta_pct(to_float(base.get("coverage@10")),
                                 to_float(cl.get("coverage@10"))))
        L.append(f"{cl_model} & CE & " + " & ".join(fmt_pct(v) for v in row)
                 + f" & {fmt_pct(_row_mean(row))} \\\\")

    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


# ---------- Consolidation ----------

def emit_consolidation(data, filename="table_consolidation.tex"):
    """Consolidation table."""
    all_rows = sum((data[k] for k in ["stage1","stage2","stage3","stage4","stage5","stage6"]
                    if k in data), [])
    cov_rows = data.get("coverage", [])

    def avg_delta(base_model, cl_model, loss, metric, rows_):
        vals = []
        for ds in DATASETS:
            base = find_row(rows_, base_model, loss, "none", ds)
            if base is None:
                continue
            b = to_float(base.get(metric))
            if b is None or b == 0:
                continue
            for cl_type in ["info_nce", "dcl"]:
                cl = find_row(rows_, cl_model, loss, cl_type, ds)
                if cl is not None:
                    d = delta_pct(b, to_float(cl.get(metric)))
                    if d is not None:
                        vals.append(d)
        return mean(vals)

    def avg_cov_delta(base_model, cl_model, loss):
        vals = []
        for ds in DATASETS:
            base = find_row(cov_rows, base_model, loss, "none", ds)
            if base is None:
                continue
            b = to_float(base.get("coverage@10"))
            if b is None or b == 0:
                continue
            for cl_type in ["info_nce", "dcl"]:
                cl = find_row(cov_rows, cl_model, loss, cl_type, ds)
                if cl is not None:
                    d = delta_pct(b, to_float(cl.get("coverage@10")))
                    if d is not None:
                        vals.append(d)
        return mean(vals)

    BACKBONES = [
        ("Mamba4Rec", "Mamba4Rec", "Mamba4Rec_CL"),
        ("SIGMA",     "SIGMA",     "SIGMA_CL"),
    ]

    L = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Consolidation — Data-level CL effect on accuracy, geometry, and coverage (mean $\Delta\%$ across four datasets). The three columns move independently across configurations, indicating that accuracy, geometry, and coverage are distinct failure modes.}",
        r"\label{tab:consolidation}",
        r"\small",
        r"\setlength{\tabcolsep}{10pt}",
        r"\begin{tabular}{l l c c c}",
        r"\toprule",
        r"Backbone & Loss & $\Delta$ NDCG@10 & $\Delta$ Isotropy & $\Delta$ Coverage@10 \\",
        r"\midrule",
    ]
    for backbone, base_model, cl_model in BACKBONES:
        for loss in ["CE", "BPR"]:
            ndcg = avg_delta(base_model, cl_model, loss, "ndcg@10", all_rows)
            iso  = avg_delta(base_model, cl_model, loss, "isotropy", all_rows)
            cov  = avg_cov_delta(base_model, cl_model, loss)
            L.append(f"{backbone} & {loss} & {fmt_pct(ndcg)} & {fmt_pct(iso)} & {fmt_pct(cov)} \\\\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(filename, "w") as f:
        f.write("\n".join(L))
    print(f"Wrote {filename}\n")


# ---------- Main ----------

def main():
    data = load_all()
    ml_rows = load_model_level()

    print("=== Emitting results tables + figure ===\n")
    emit_rq1_compact(data, ml_rows)
    emit_rq2_compact(data, ml_rows)
    emit_rq3_compact(data, ml_rows)
    emit_rq4_compact(data)
    emit_rq4_stability_table(data)
    emit_rq4_loss_curves(data)
    emit_rq5_compact(data)
    emit_consolidation(data)

    print("Done.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[info] stage1_results_full.csv: extended header from 25 to 28 columns
[info] avg: 12 rows loaded from test_results_amazon_videogames.csv
[info] toys: 12 rows loaded from test_results_amazon_toys_and_games.csv
[info] ml1m: 12 rows loaded from test_results_movielens_1m.csv
[info] lastfm: 12 rows loaded from test_results_lastfm_1k.csv
[info] model-level total: 48 rows

=== Emitting results tables + figure ===

Wrote table_rq1.tex

Wrote table_rq2.tex

Wrote table_rq3.tex

Wrote table_rq4.tex

Wrote table_rq4_stability.tex

Wrote fig_rq4_loss_curves.pdf

Wrote table_rq5.tex

Wrote table_consolidation.tex

Done.
